# Pose Correction — Temporal Fusion Transformer (TFT-inspired)

This notebook trains a **Temporal Fusion Transformer**–style model (Lim et al., *Temporal Fusion Transformers for Interpretable Multi-horizon Time Series Forecasting*, 2020) adapted to **sequence → vector regression** for final joint displacement.

**Task (same as other pose notebooks)**
- **Inputs**: pose sequence (`T × 12 × C` from dataset metadata; `C = n_coords`, usually 3, or **2** if data was built with `--drop-z`) + workout class features (broadcast over time).
- **Target**: final displacement vector (`12 × C`, i.e. **36** or **24**).

**Architecture (TFT ingredients, adapted)**
1. **Static branch**: workout class vector from the sequence → **gated residual network (GRN)** → context broadcast to every timestep.
2. **Temporal branch**: per-timestep pose + **normalized time** (known covariate) → `TimeDistributed` projection → **GRN**.
3. **Fusion**: add static context, **LayerNorm**, then **stacked LSTM** for local temporal patterns.
4. **Temporal fusion**: **multi-head self-attention** + position-wise FFN with residuals (Transformer-style block on top of LSTM states).
5. **Head**: global average pooling over time → small MLP → **`12×C` outputs** (same width as `y`).

Full TFT variable-selection softmax over *individual raw series* and per-horizon **interpretable** attention (one key per head) are simplified here for a fixed-length pose window; the notebook keeps **GRN gating**, **static–temporal fusion**, **LSTM + attention**, which are the core TFT ideas for mixing covariates and long-range dependence.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import callbacks, layers, models

np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")


def r2_score_np(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - np.mean(y_true, axis=0, keepdims=True)) ** 2))
    return 1.0 - ss_res / (ss_tot + 1e-12)


## Load displacement training data

In [ ]:
data_path = Path("../../Data/output_displacement/training_data_displacement.npz")
metadata_path = Path("../../Data/output_displacement/training_data_displacement_metadata.json")

with open(metadata_path, "r") as f:
    metadata = json.load(f)

print("Dataset metadata:")
print(f"  n_samples: {metadata['n_samples']}")
print(f"  sequence_length: {metadata['sequence_length']}")
print(f"  n_landmarks: {metadata['n_landmarks']}")
print(f"  X_pose_flat_dim: {metadata['X_pose_flat_dim']}")
print(f"  X_class_dim: {metadata['X_class_dim']}")
print(f"  y_final_disp_flat_dim: {metadata['y_final_disp_flat_dim']}")
print(f"  class_encoding: {metadata['class_encoding']}")
n_coords_meta = int(metadata.get("n_coords", 3))
print(f"  n_coords: {n_coords_meta}  (pose per frame = {int(metadata['n_landmarks'])} * {n_coords_meta})")

data = np.load(data_path, allow_pickle=True)
print(f"\n.npz keys: {list(data.keys())}")

X = data["X"].astype(np.float32)
y = data["y"].astype(np.float32)
y_class = data["y_class"].astype(np.int64)
class_names = data["class_names"]

print(f"\nX: {X.shape}, y: {y.shape}, y_class: {y_class.shape}, classes: {len(class_names)}")


## Preprocess (sequence tensor + normalization)

Same pipeline as `Transformer_pose_correction.ipynb` / `BiGRU_MHA_pose_correction.ipynb`: reshape pose to `(T, 12 * n_coords)` using `n_coords` from metadata (24 or 36 pose values per timestep), broadcast class over time, normalize `X`, standardize `y`.

In [ ]:
N = int(X.shape[0])
T = int(metadata["sequence_length"])
pose_flat_dim = int(metadata["X_pose_flat_dim"])
class_dim = int(metadata["X_class_dim"])

X_pose_flat = X[:, :pose_flat_dim]
X_class_feat = X[:, pose_flat_dim : pose_flat_dim + class_dim]

X_pose_seq = X_pose_flat.reshape(N, T, -1)
n_coords = int(metadata.get("n_coords", 3))
n_landmarks = int(metadata["n_landmarks"])
pose_feats_per_step = n_landmarks * n_coords
if X_pose_seq.shape[2] != pose_feats_per_step:
    raise ValueError(
        f"Expected {pose_feats_per_step} pose features per timestep "
        f"(n_landmarks={n_landmarks} * n_coords={n_coords}), got {X_pose_seq.shape[2]}"
    )

X_class_seq = np.repeat(X_class_feat[:, None, :], T, axis=1)
X_seq = np.concatenate([X_pose_seq, X_class_seq], axis=2)

print(f"X_seq: {X_seq.shape}  y: {y.shape}")

X_seq = np.nan_to_num(X_seq, nan=0.0, posinf=0.0, neginf=0.0)
y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)

X_mean = np.mean(X_seq, axis=(0, 1), keepdims=True)
X_std = np.std(X_seq, axis=(0, 1), keepdims=True) + 1e-8
X_normalized = (X_seq - X_mean) / X_std

y_mean = np.mean(y, axis=0, keepdims=True)
y_std = np.std(y, axis=0, keepdims=True) + 1e-8
y_standardized = (y - y_mean) / y_std

print(f"X_norm ready. y_standardized: {y_standardized.shape}")


## Train / test split (stratified by workout class)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, y_class_train, y_class_test = train_test_split(
    X_normalized,
    y_standardized,
    y_class,
    test_size=0.2,
    random_state=42,
    stratify=y_class,
)

print(f"train: {X_train.shape[0]}, test: {X_test.shape[0]}")
print(f"seq: {X_train.shape[1:]} → target dim {y_train.shape[1]}")


## TFT-inspired model (GRN + static context + LSTM + attention)

The functional model splits the normalized sequence into **pose** `(T, 12 * n_coords)` and **static class** `(C,)` from `t=0` (class is identical at all timesteps). **Known time** in `[0, 1]` is concatenated as an extra per-step feature.

In [ ]:
class GatedResidualNetwork(layers.Layer):
    """TFT-style GRN: ELU MLP with sigmoid gate and skip (Lim et al., 2020)."""

    def __init__(self, width: int, dropout: float = 0.1, **kwargs):
        super().__init__(**kwargs)
        self.width = int(width)
        self.dropout = float(dropout)
        self.fc1 = layers.Dense(self.width, activation="elu")
        self.fc2 = layers.Dense(self.width)
        self.gate = layers.Dense(self.width, activation="sigmoid")
        self.norm = layers.LayerNormalization(epsilon=1e-6)
        self.drop = layers.Dropout(self.dropout)
        self.proj = None

    def build(self, input_shape):
        last = int(input_shape[-1])
        if last != self.width:
            self.proj = layers.Dense(self.width)
        super().build(input_shape)

    def call(self, inputs, training=False):
        skip = self.proj(inputs) if self.proj is not None else inputs
        h = self.fc1(inputs)
        h = self.fc2(h)
        h = self.drop(h, training=training)
        g = self.gate(inputs)
        return self.norm(skip + g * h)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"width": self.width, "dropout": self.dropout})
        return cfg


class TemporalFusionBlock(layers.Layer):
    """Self-attention + pointwise FFN with residuals and layer norms."""

    def __init__(
        self,
        d_model: int,
        num_heads: int,
        dff: int,
        dropout: float = 0.1,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.d_model = int(d_model)
        self.num_heads = int(num_heads)
        self.dff = int(dff)
        self.dropout = float(dropout)
        key_dim = max(1, self.d_model // self.num_heads)
        self.mha = layers.MultiHeadAttention(
            num_heads=self.num_heads,
            key_dim=key_dim,
            dropout=self.dropout,
        )
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.ffn = keras.Sequential(
            [
                layers.Dense(self.dff, activation="gelu"),
                layers.Dropout(self.dropout),
                layers.Dense(self.d_model),
                layers.Dropout(self.dropout),
            ]
        )

    def build(self, input_shape):
        """Keras 3: ensure MHA is built; avoids load_model marking the block built without attention."""
        super().build(input_shape)
        self.mha.build(input_shape, input_shape, input_shape)

    def call(self, x, training=False):
        attn = self.mha(x, x, training=training)
        x = self.norm1(x + attn)
        f = self.ffn(x, training=training)
        return self.norm2(x + f)

    def get_config(self):
        cfg = super().get_config()
        cfg.update(
            {
                "d_model": self.d_model,
                "num_heads": self.num_heads,
                "dff": self.dff,
                "dropout": self.dropout,
            }
        )
        return cfg


def build_tft_pose_model(
    n_timesteps: int,
    n_pose: int,
    n_class: int,
    out_dim: int,
    *,
    d_model: int = 192,
    lstm_units: int = 192,
    num_heads: int = 4,
    dff: int = 384,
    n_fusion_blocks: int = 2,
    dropout: float = 0.12,
) -> models.Model:
    """Single input (T, n_pose + n_class); splits inside graph."""
    inputs = layers.Input(shape=(n_timesteps, n_pose + n_class), name="seq")

    pose = layers.Lambda(lambda t: t[:, :, :n_pose], name="pose_slice")(inputs)
    static_vec = layers.Lambda(lambda t: t[:, 0, n_pose:], name="static_class")(inputs)

    def normalized_time(x):
        b = tf.shape(x)[0]
        tlen = tf.shape(x)[1]
        u = tf.linspace(0.0, 1.0, tlen)
        u = tf.reshape(u, [1, tlen, 1])
        return tf.tile(u, [b, 1, 1])

    time_feat = layers.Lambda(normalized_time, name="time_known")(pose)
    temporal_raw = layers.Concatenate(axis=-1, name="pose_time")([pose, time_feat])

    static_ctx = GatedResidualNetwork(d_model, dropout=dropout, name="static_grn")(static_vec)
    static_ctx_t = layers.RepeatVector(n_timesteps, name="static_broadcast")(static_ctx)

    temporal_emb = layers.TimeDistributed(
        layers.Dense(d_model, use_bias=True),
        name="temporal_proj",
    )(temporal_raw)
    temporal_emb = GatedResidualNetwork(d_model, dropout=dropout, name="temporal_grn")(temporal_emb)

    x = layers.Add(name="static_temporal_add")([temporal_emb, static_ctx_t])
    x = layers.LayerNormalization(epsilon=1e-6, name="post_fusion_ln")(x)

    x = layers.LSTM(
        lstm_units,
        return_sequences=True,
        dropout=dropout,
        recurrent_dropout=0.0,
        name="lstm_1",
    )(x)
    x = layers.LSTM(
        lstm_units,
        return_sequences=True,
        dropout=dropout,
        recurrent_dropout=0.0,
        name="lstm_2",
    )(x)

    if lstm_units != d_model:
        x = layers.Conv1D(d_model, kernel_size=1, padding="same", name="lstm_to_d")(x)
    x = layers.LayerNormalization(epsilon=1e-6, name="post_lstm_ln")(x)

    for i in range(n_fusion_blocks):
        x = TemporalFusionBlock(
            d_model=d_model,
            num_heads=num_heads,
            dff=dff,
            dropout=dropout,
            name=f"fusion_block_{i+1}",
        )(x)

    x = layers.GlobalAveragePooling1D(name="gap")(x)
    x = layers.Dense(128, activation="gelu", name="head_1")(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(64, activation="gelu", name="head_2")(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(out_dim, activation="linear", name="disp")(x)

    return models.Model(inputs, outputs, name="tft_pose_correction")


n_timesteps = X_train.shape[1]
n_features = X_train.shape[2]
n_pose = int(metadata["n_landmarks"]) * int(metadata.get("n_coords", 3))
n_class = n_features - n_pose
out_dim = y_train.shape[1]

if n_class != class_dim:
    raise ValueError(f"Expected n_class {class_dim}, got {n_class} from features")

model = build_tft_pose_model(
    n_timesteps,
    n_pose,
    n_class,
    out_dim,
    d_model=192,
    lstm_units=192,
    num_heads=4,
    dff=384,
    n_fusion_blocks=2,
    dropout=0.12,
)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=8e-4),
    loss="mse",
    metrics=[
        keras.metrics.MeanSquaredError(name="mse"),
        keras.metrics.MeanAbsoluteError(name="mae"),
    ],
)

model.summary()


## Train (early stopping, checkpoint, LR reduction)

In [ ]:
early_stopping = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True,
    verbose=1,
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1,
)

models_dir = Path("models")
models_dir.mkdir(exist_ok=True)

ckpt = callbacks.ModelCheckpoint(
    filepath=str(models_dir / "tft_pose_correction_best.keras"),
    monitor="val_loss",
    save_best_only=True,
    verbose=1,
)

batch_size = 32
epochs = 200

print("Training TFT pose correction model...")
history = model.fit(
    X_train,
    y_train,
    batch_size=batch_size,
    epochs=epochs,
    validation_data=(X_test, y_test),
    callbacks=[early_stopping, reduce_lr, ckpt],
    verbose=1,
)
print("Done.")


## Training curves (MSE + MAE on standardized targets)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["mse"], label="train")
axes[0].plot(history.history["val_mse"], label="val")
axes[0].set_title("MSE (standardized y)")
axes[0].set_xlabel("epoch")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history["mae"], label="train")
axes[1].plot(history.history["val_mae"], label="val")
axes[1].set_title("MAE (standardized y)")
axes[1].set_xlabel("epoch")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Evaluate on original displacement scale

In [ ]:
y_pred_std = model.predict(X_test, verbose=0).astype(np.float32)

y_pred = y_pred_std * y_std + y_mean
y_true = y_test * y_std + y_mean

mse = float(np.mean((y_true - y_pred) ** 2))
rmse = float(np.sqrt(mse))
mae = float(np.mean(np.abs(y_true - y_pred)))
r2 = r2_score_np(y_true, y_pred)

print(f"MSE:  {mse:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"MAE:  {mae:.6f}")
print(f"R2:   {r2:.6f}")

n_coords = int(metadata.get("n_coords", 3))
n_landmarks = int(metadata["n_landmarks"])

err = np.abs(y_true - y_pred)
err_joints = err.reshape(-1, n_landmarks, n_coords)
joint_mae = err_joints.mean(axis=0)

print(f"\nPer-joint MAE (n_coords={n_coords}):")
for j in range(n_landmarks):
    if n_coords == 3:
        print(f"  joint {j:02d}: {joint_mae[j,0]:.6f}, {joint_mae[j,1]:.6f}, {joint_mae[j,2]:.6f}")
    else:
        print(f"  joint {j:02d}: {joint_mae[j,0]:.6f}, {joint_mae[j,1]:.6f}")


## Error visualization

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(joint_mae.mean(axis=1), marker="o")
plt.title("Per-joint MAE (mean over axes)")
plt.xlabel("Joint index")
plt.ylabel("MAE")
plt.grid(True)
plt.show()

n_coords = int(metadata.get("n_coords", 3))

plt.figure(figsize=(8, 4))
plt.imshow(joint_mae, aspect="auto")
plt.title("MAE heatmap (joint × axis)")
plt.xlabel("axis")
plt.ylabel("joint")
plt.xticks(range(n_coords), ["x", "y", "z"][:n_coords])
plt.colorbar(label="MAE")
plt.tight_layout()
plt.show()


## Example predictions

In [ ]:
rng = np.random.default_rng(42)
idxs = rng.choice(len(X_test), size=5, replace=False)

n_coords = int(metadata.get("n_coords", 3))
n_landmarks = int(metadata["n_landmarks"])

for k, i in enumerate(idxs, start=1):
    yt = y_true[i].reshape(n_landmarks, n_coords)
    yp = y_pred[i].reshape(n_landmarks, n_coords)
    ae = np.abs(yt - yp)
    print(f"\nExample {k} (test idx={int(i)})")
    print(f"  mean |err|: {ae.mean():.6f}  max |err|: {ae.max():.6f}")
    for j in range(min(3, n_landmarks)):
        if n_coords == 3:
            print(
                f"  j{j:02d} true {yt[j,0]: .4f},{yt[j,1]: .4f},{yt[j,2]: .4f} | "
                f"pred {yp[j,0]: .4f},{yp[j,1]: .4f},{yp[j,2]: .4f}"
            )
        else:
            print(
                f"  j{j:02d} true {yt[j,0]: .4f},{yt[j,1]: .4f} | "
                f"pred {yp[j,0]: .4f},{yp[j,1]: .4f}"
            )


## Save final weights (in addition to best checkpoint)

In [ ]:
final_path = models_dir / "tft_pose_correction_final.keras"
model.save(final_path)
print(f"Saved: {final_path}")
